In [1]:
from __future__ import annotations

import asyncio
import json
import os
import re
import time
from pathlib import Path
from typing import Dict, List, Optional
from urllib.parse import urljoin

import aiohttp
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from vllm import LLM, SamplingParams

/scratch4/home/akrik/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Config ───────────────────────────────────────────────────────────────────
INDEX_URL          = "https://man7.org/linux/man-pages/dir_all_alphabetic.html"
OUTPUT_JSONL_PATH  = Path("/scratch4/home/akrik/NTILC/data/man/raw_ai.jsonl")
ERRORS_PATH        = Path("/scratch4/home/akrik/NTILC/data/man/raw_errors_ai.json")
MODEL_ID           = "Qwen/Qwen3-30B-A3B-Instruct-2507"

CUDA_VISIBLE_DEVICES  = "5,6"
TENSOR_PARALLEL_SIZE  = 2

MAX_PAGES: Optional[int] = None
REQUEST_TIMEOUT        = 20.0
TEMP                   = 0.0
MAX_NEW_TOKENS         = 2048
MAX_MODEL_LEN          = 194400
HTTP_CONCURRENCY       = 32
INFERENCE_BATCH_SIZE   = 256
RESUME_FROM_DISK       = True

SECTION_1_PATTERN = re.compile(r"\(1\)$")
DEFAULT_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/128 Safari/537.36"
    )
}

# Must be set before vLLM loads
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES

In [3]:
# ── Prompts ───────────────────────────────────────────────────────────────────
EXTRACTION_SYSTEM_PROMPT = """You extract structured JSON from a Linux man-page HTML body.
Return strict JSON only. Do not include markdown or explanation.
Use this schema exactly:
{
  \"name\": \"string\",
  \"one_line\": \"string\",
  \"description\": \"string\",
  \"invocation\": \"string\",
  \"options\": [
    {
      \"flags\": [\"string\"],
      \"arg\": \"string\",
      \"description\": \"string\"
    }
  ]
}
If a field is unknown, use empty string/list.
Only include real options present in the HTML.
"""

EXAMPLE_OUTPUT = {
    "name": "ls",
    "one_line": "list directory contents",
    "description": "List information about files.",
    "invocation": "ls [OPTION]... [FILE]...",
    "options": [
        {"flags": ["-a", "--all"], "arg": "", "description": "do not ignore entries starting with ."}
    ],
}


def build_prompt(*, source_url: str, html: str) -> str:
    return (
        f"SYSTEM PROMPT:\n{EXTRACTION_SYSTEM_PROMPT}\n\n"
        f"EXAMPLE OUTPUT:\n{json.dumps(EXAMPLE_OUTPUT, ensure_ascii=False, indent=2)}\n\n"
        f"SOURCE URL:\n{source_url}\n\n"
        "HTML BODY:\n"
        "```html\n"
        f"{html}\n"
        "```\n"
    )

In [ ]:
# ── Step 1: Collect links ─────────────────────────────────────────────────────
print("Fetching index ...")
resp = requests.get(INDEX_URL, timeout=30, headers=DEFAULT_HEADERS)
soup = BeautifulSoup(resp.text, "html.parser")

all_links: List[str] = sorted({
    urljoin(INDEX_URL, a["href"])
    for a in soup.find_all("a")
    if a.get_text(strip=True) and a.get("href") and SECTION_1_PATTERN.search(a.get_text(strip=True))
})

if MAX_PAGES:
    all_links = all_links[:MAX_PAGES]

print(f"Found {len(all_links)} section-1 links")
all_links[:5]

Fetching index ...


In [ ]:
# ── Step 2: Resume ────────────────────────────────────────────────────────────
def load_done_urls(path: Path) -> set[str]:
    done = set()
    if path.exists():
        with path.open() as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    if "source_url" in obj:
                        done.add(obj["source_url"])
                except json.JSONDecodeError:
                    pass
    return done


done_urls: set[str] = load_done_urls(OUTPUT_JSONL_PATH) if RESUME_FROM_DISK else set()
todo_links = [u for u in all_links if u not in done_urls]
print(f"{len(done_urls)} already done — {len(todo_links)} remaining")

In [ ]:
# ── Step 3: Fetch all HTML concurrently ───────────────────────────────────────
async def fetch_one(
    session: aiohttp.ClientSession,
    url: str,
    semaphore: asyncio.Semaphore,
) -> tuple[str, str | None]:
    async with semaphore:
        try:
            async with session.get(url, timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT)) as r:
                r.raise_for_status()
                return url, await r.text()
        except Exception as exc:
            print(f"[fetch error] {url}: {exc}")
            return url, None


async def fetch_all(urls: List[str]) -> Dict[str, str | None]:
    semaphore = asyncio.Semaphore(HTTP_CONCURRENCY)
    connector = aiohttp.TCPConnector(limit=HTTP_CONCURRENCY, ssl=False)
    async with aiohttp.ClientSession(headers=DEFAULT_HEADERS, connector=connector) as session:
        tasks = [fetch_one(session, url, semaphore) for url in urls]
        results = {}
        for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Fetching HTML"):
            url, html = await coro
            results[url] = html
    return results


try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

html_map = asyncio.run(fetch_all(todo_links))
print(f"Fetched {sum(v is not None for v in html_map.values())} pages successfully")

In [ ]:
# ── Step 4: Build prompts ─────────────────────────────────────────────────────
pairs: List[tuple[str, str]] = []
fetch_errors: List[dict] = []

for url in todo_links:
    html = html_map.get(url)
    if html is None:
        fetch_errors.append({"url": url, "error": "fetch_failed"})
        continue
    pairs.append((url, build_prompt(source_url=url, html=html)))

print(f"Built {len(pairs)} prompts — {len(fetch_errors)} fetch errors")

In [ ]:
# ── Step 5: Load model with vLLM ─────────────────────────────────────────────
# NOTE: To change CUDA_VISIBLE_DEVICES or max_model_len you must restart the kernel.
llm = LLM(
    model=MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    dtype="bfloat16",
    trust_remote_code=True,
    gpu_memory_utilization=0.92,
    enable_chunked_prefill=True,
    max_num_batched_tokens=32768,
    max_model_len=MAX_MODEL_LEN,
)

sampling_params = SamplingParams(
    temperature=TEMP,
    max_tokens=MAX_NEW_TOKENS,
)

In [ ]:
# ── Step 6: Filter prompts that are too long ──────────────────────────────────
# Must run AFTER Step 5 since we need llm.get_tokenizer()
MAX_PROMPT_TOKENS = MAX_MODEL_LEN - MAX_NEW_TOKENS

tokenizer = llm.get_tokenizer()

errors: List[dict] = list(fetch_errors)  # seed with fetch errors
filtered_pairs, skipped = [], []

for url, prompt in tqdm(pairs, desc="Checking token lengths"):
    n = len(tokenizer.encode(prompt))
    if n <= MAX_PROMPT_TOKENS:
        filtered_pairs.append((url, prompt))
    else:
        print(f"[skip] {url} — {n:,} tokens")
        skipped.append({"url": url, "error": f"prompt_too_long_{n}_tokens"})

errors.extend(skipped)
pairs = filtered_pairs
print(f"{len(pairs)} within limit, {len(skipped)} skipped")

In [ ]:
# ── Step 7: Inference + save ──────────────────────────────────────────────────
def parse_json_output(raw: str) -> dict:
    raw = raw.strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    if m:
        raw = m.group(0)
    return json.loads(raw)


OUTPUT_JSONL_PATH.parent.mkdir(parents=True, exist_ok=True)
ERRORS_PATH.parent.mkdir(parents=True, exist_ok=True)

total_processed = 0

with OUTPUT_JSONL_PATH.open("a") as out_file:
    for batch_start in tqdm(range(0, len(pairs), INFERENCE_BATCH_SIZE), desc="Inference batches"):
        batch         = pairs[batch_start : batch_start + INFERENCE_BATCH_SIZE]
        urls_batch    = [u for u, _ in batch]
        prompts_batch = [p for _, p in batch]

        t0 = time.time()
        outputs = llm.generate(prompts_batch, sampling_params)
        elapsed = time.time() - t0
        print(f"  batch {batch_start // INFERENCE_BATCH_SIZE}: "
              f"{len(batch)} pages in {elapsed:.1f}s ({elapsed/len(batch):.2f}s/page)")

        for url, output in zip(urls_batch, outputs):
            raw_text = output.outputs[0].text
            try:
                parsed = parse_json_output(raw_text)
                parsed["source_url"] = url
                out_file.write(json.dumps(parsed, ensure_ascii=False) + "\n")
            except Exception as exc:
                errors.append({"url": url, "error": str(exc), "raw": raw_text[:2000]})

        out_file.flush()
        total_processed += len(batch)

with ERRORS_PATH.open("w") as f:
    json.dump(errors, f, ensure_ascii=False, indent=2)

print(f"\nDone. {total_processed} processed, {len(errors)} errors.")
print(f"Results → {OUTPUT_JSONL_PATH}")
print(f"Errors  → {ERRORS_PATH}")